# Disease Prediction - Model Training

This notebook is used to prepare the cleaned dataset, train machine learning models, evaluate their performance, and save the trained models.

## Import Required Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Load Cleaned Dataset

In [2]:
data_path = "C:\\Users\\PC\\Desktop\\Disease-Prediction-ML\\dataset\\cleaned_disease_dataset.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (189647, 378)


## Separate Features and Target

In [3]:
X = df.drop("diseases", axis=1)
y = df["diseases"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (189647, 377)
Target shape: (189647,)


## Encode Disease Labels

The disease names are converted into numerical labels so that machine learning models can process the target values.

In [4]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Number of disease classes:", len(label_encoder.classes_))
print("Encoded target shape:", y_encoded.shape)

Number of disease classes: 773
Encoded target shape: (189647,)


## Check Disease Class Distribution

In [5]:
class_counts = y.value_counts()

print("Minimum samples in a disease:", class_counts.min())
print("Maximum samples in a disease:", class_counts.max())
print("Median samples per disease:", class_counts.median())

Minimum samples in a disease: 1
Maximum samples in a disease: 1219
Median samples per disease: 77.0


## Identify Rare Disease Classes


In [6]:
rare_classes = class_counts[class_counts < 2]

print("Number of diseases with only 1 sample:", len(rare_classes))

Number of diseases with only 1 sample: 46


## Remove Classes With Only One Sample

In [7]:
# Keep only diseases having at least 2 samples
df_model = df[df["diseases"].map(class_counts) >= 2].copy()

print("Original dataset shape:", df.shape)
print("Dataset shape after removing singleton classes:", df_model.shape)
print("Number of diseases remaining:", df_model["diseases"].nunique())

Original dataset shape: (189647, 378)
Dataset shape after removing singleton classes: (189601, 378)
Number of diseases remaining: 727


## Prepare Features and Target

In [8]:
X = df_model.drop("diseases", axis=1)
y = df_model["diseases"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (189601, 377)
Target shape: (189601,)


## Encode Updated Disease Labels

In [9]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Number of disease classes:", len(label_encoder.classes_))
print("Encoded target shape:", y_encoded.shape)

Number of disease classes: 727
Encoded target shape: (189601,)


## Train-Test Split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (151680, 377)
Testing features: (37921, 377)
Training target: (151680,)
Testing target: (37921,)


## Create Models Folder

In [ ]:
import os
import joblib

models_dir = "C:\\Users\\PC\\Desktop\\Disease-Prediction-ML\\models"

os.makedirs(models_dir, exist_ok=True)

joblib.dump(label_encoder, os.path.join(models_dir, "label_encoder.pkl"))

print("Label encoder saved successfully!")

## Train Softmax Regression

In [ ]:
softmax_model = LogisticRegression(
    C=10,
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)

softmax_model.fit(X_train, y_train)

print("Softmax Regression training completed!")

y_pred_softmax = softmax_model.predict(X_test)

softmax_accuracy = accuracy_score(y_test, y_pred_softmax)

print("Softmax Regression Accuracy:", softmax_accuracy)
print("Softmax Regression Accuracy (%):", softmax_accuracy * 100)

joblib.dump(softmax_model, os.path.join(models_dir, "softmax_regression_model.pkl"))

print("Softmax Regression model saved successfully!")

## Train Random Forest

In [ ]:
random_forest_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features=0.3,
    bootstrap=True,
    random_state=42,
    n_jobs=1
)

random_forest_model.fit(X_train, y_train)

print("Random Forest training completed!")

y_pred_rf = random_forest_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", rf_accuracy)
print("Random Forest Accuracy (%):", rf_accuracy * 100)

joblib.dump(random_forest_model, os.path.join(models_dir, "random_forest_model.pkl"))

print("Random Forest model saved successfully!")

## Train XGBoost

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

print("XGBoost training completed!")

y_pred_xgb = xgb_model.predict(X_test)

xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

print("XGBoost Accuracy:", xgb_accuracy)
print("XGBoost Accuracy (%):", xgb_accuracy * 100)

joblib.dump(xgb_model, os.path.join(models_dir, "xgboost_model.pkl"))

print("XGBoost model saved successfully!")